In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
! unzip /content/drive/MyDrive/png.zip

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  inflating: all/abnormal/S(83-167)0058.png  
  inflating: all/abnormal/S(83-167)0059.png  
  inflating: all/abnormal/S(83-167)0060.png  
  inflating: all/abnormal/S(83-167)0061.png  
  inflating: all/abnormal/S(83-167)0062.png  
  inflating: all/abnormal/S(83-167)0063.png  
  inflating: all/abnormal/S(83-167)0064.png  
  inflating: all/abnormal/S(83-167)0065.png  
  inflating: all/abnormal/S(83-167)0066.png  
  inflating: all/abnormal/S(83-167)0067.png  
  inflating: all/abnormal/S(83-167)0068.png  
  inflating: all/abnormal/S(83-167)0069.png  
  inflating: all/abnormal/S(83-167)0070.png  
  inflating: all/abnormal/S(83-167)0071.png  
  inflating: all/abnormal/S(83-167)0072.png  
  inflating: all/abnormal/S(83-167)0073.png  
  inflating: all/abnormal/S(83-167)0074.png  
  inflating: all/abnormal/S(83-167)0075.png  
  inflating: all/abnormal/S(83-167)0076.png  
  inflating: all/abnormal/S(83-167)0077.png  
  inflating: all/abnormal/S(83-16

In [3]:
import json
import numpy as np
from PIL import Image, ImageDraw
import os

# JSON dosyasını yükle
with open('annotations.json', 'r') as file:
    data = json.load(file)

# Görüntülerin kaydedildiği dizin
image_dir = '/content/all/abnormal/'

# Orijinal görüntülerin kaydedileceği dizin
originals_dir = '/content/drive/MyDrive/originals/'

# Maskelerin kaydedileceği dizin
masks_dir = '/content/drive/MyDrive/masks/'

# Gerekirse klasörleri oluştur
os.makedirs(originals_dir, exist_ok=True)
os.makedirs(masks_dir, exist_ok=True)

for image_info in data['images']:
    # İlgili annotasyonları bul
    annotations = [anno for anno in data['annotations'] if anno['image_id'] == image_info['id']]

    # Eğer görüntü için annotasyon yoksa, döngüde sonraki görüntüye geç
    if not annotations:
        continue

    # Görüntü dosyasını aç
    image_path = os.path.join(image_dir, image_info['file_name'])
    image = Image.open(image_path)
    width, height = image.size

    # Orijinal görüntüyü kaydet
    original_save_path = os.path.join(originals_dir, image_info['file_name'])
    image.save(original_save_path)

    # Binary maske için boş bir görüntü oluştur
    mask = Image.new('L', (width, height), 0)
    draw = ImageDraw.Draw(mask)

    for annotation in annotations:
        # 'segmentation' bilgisini kullanarak maskeyi çiz
        for segment in annotation['segmentation']:
            # Segmentasyon noktalarını çiftler halinde grupla (x, y)
            polygon = [(segment[i], segment[i+1]) for i in range(0, len(segment), 2)]
            draw.polygon(polygon, outline=255, fill=255)  # Beyaz dolgu ve çerçeve

    # Maskeyi kaydet
    mask_save_path = os.path.join(masks_dir, image_info['file_name'].replace('.png', '_mask.png'))
    mask.save(mask_save_path)

    print(f"Orijinal ve maske kaydedildi: {original_save_path}, {mask_save_path}")

Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(60-107)0018.png, /content/drive/MyDrive/masks/(60-107)0018_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(60-107)0019.png, /content/drive/MyDrive/masks/(60-107)0019_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(60-107)0020.png, /content/drive/MyDrive/masks/(60-107)0020_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(60-107)0021.png, /content/drive/MyDrive/masks/(60-107)0021_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(60-107)0022.png, /content/drive/MyDrive/masks/(60-107)0022_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(74-141)0049.png, /content/drive/MyDrive/masks/(74-141)0049_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(74-141)0050.png, /content/drive/MyDrive/masks/(74-141)0050_mask.png
Orijinal ve maske kaydedildi: /content/drive/MyDrive/originals/(74-141)0051.

In [4]:
import cv2
import numpy as np
import os


def goruntu_isle_ve_kaydet(dosya_yolu, hedef_klasor):
    # Görüntüyü yükle
    image = cv2.imread(dosya_yolu)

    # Görüntüyü BGR'dan griye çevir (eğer zaten siyah beyaz değilse)
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Gri görüntüyü tekrar BGR'a çevir ki renk ekleyebilelim
    color_image = cv2.cvtColor(gray_image, cv2.COLOR_GRAY2BGR)

    # Beyaz olan pikselleri bul (255 olanlar)
    white_pixels = np.where((color_image[:, :, 0] == 255) & (color_image[:, :, 1] == 255) & (color_image[:, :, 2] == 255))

    # Bulunan beyaz pikselleri kırmızı yap
    color_image[white_pixels] = [0, 0, 255]

    # İşlenmiş görüntüyü hedef klasöre kaydet
    cv2.imwrite(os.path.join(hedef_klasor, os.path.basename(dosya_yolu)), color_image)

# Kaynak klasör yolu
kaynak_klasor = '/content/drive/MyDrive/masks'

# Hedef klasör yolu
hedef_klasor = '/content/drive/MyDrive/red_masks'

# Hedef klasör yoksa oluştur
if not os.path.exists(hedef_klasor):
    os.makedirs(hedef_klasor)

# Kaynak klasördeki tüm dosyaları listele ve işle
for dosya in os.listdir(kaynak_klasor):
    dosya_yolu = os.path.join(kaynak_klasor, dosya)
    # Görüntüyü işle ve hedef klasöre kaydet
    goruntu_isle_ve_kaydet(dosya_yolu, hedef_klasor)


In [8]:
import os

import cv2


input_dir = '/content/drive/MyDrive/tmp/red_masks'
output_dir = '/content/drive/MyDrive/tmp/labels'

for j in os.listdir(input_dir):
    image_path = os.path.join(input_dir, j)
    # load the binary mask and get its contours
    mask = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)

    H, W = mask.shape
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # convert the contours to polygons
    polygons = []
    for cnt in contours:
          polygon = []
          for point in cnt:
              x, y = point[0]
              polygon.append(x / W)
              polygon.append(y / H)
          polygons.append(polygon)

    # print the polygons
    with open('{}.txt'.format(os.path.join(output_dir, j)[:-4]), 'w') as f:
        for polygon in polygons:
            for p_, p in enumerate(polygon):
                if p_ == len(polygon) - 1:
                    f.write('{}\n'.format(p))
                elif p_ == 0:
                    f.write('0 {} '.format(p))
                else:
                    f.write('{} '.format(p))

        f.close()

In [10]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Ana klasör ve alt klasörlerin isimlerini tanımla
ana_klasor = '/content/drive/MyDrive/all_info/'
alt_klasorler = ['images/train', 'images/val', 'labels/train', 'labels/val']

# Gerekli klasör yapısını oluştur
for klasor in alt_klasorler:
    os.makedirs(os.path.join(ana_klasor, klasor), exist_ok=True)

# Önceden hazırlanmış fotoğraf ve maske klasörlerinin yolları
foto_klasor = '/content/drive/MyDrive/originals'
maske_klasor = '/content/drive/MyDrive/tmp/labels'

# Fotoğraf ve maske dosyalarını listele
foto_dosyalar = [dosya for dosya in os.listdir(foto_klasor) if dosya.endswith('.png')]
maske_dosyalar = [dosya for dosya in os.listdir(maske_klasor) if dosya.endswith('.txt')]

# Dosyaları eğitim ve doğrulama setlerine ayır
foto_train, foto_val = train_test_split(foto_dosyalar, test_size=0.2, random_state=42)
maske_train, maske_val = train_test_split(maske_dosyalar, test_size=0.2, random_state=42)

# Fotoğrafları uygun klasörlere kopyala
for dosya in foto_train:
    shutil.copy(os.path.join(foto_klasor, dosya), os.path.join(ana_klasor, 'images/train', dosya))

for dosya in foto_val:
    shutil.copy(os.path.join(foto_klasor, dosya), os.path.join(ana_klasor, 'images/val', dosya))

# Maskeleri uygun klasörlere kopyala
for dosya in maske_train:
    shutil.copy(os.path.join(maske_klasor, dosya), os.path.join(ana_klasor, 'labels/train', dosya))

for dosya in maske_val:
    shutil.copy(os.path.join(maske_klasor, dosya), os.path.join(ana_klasor, 'labels/val', dosya))


In [ ]:
import os

# Klasör yolu
klasor_yolu = '/content/drive/MyDrive/ImageSegmentationYOLOv8/all_info/labels/val'

# Klasördeki tüm dosyaları ve klasörleri listele
for dosya in os.listdir(klasor_yolu):
    # Yeni dosya adını oluştur (eğer dosya adında '_mask' varsa kaldır)
    yeni_dosya_adi = dosya.replace('_mask', '')

    # Eski ve yeni dosya yollarını oluştur
    eski_dosya_yolu = os.path.join(klasor_yolu, dosya)
    yeni_dosya_yolu = os.path.join(klasor_yolu, yeni_dosya_adi)

    # Dosya ismini değiştir
    os.rename(eski_dosya_yolu, yeni_dosya_yolu)

    print(f"'{dosya}' isimli dosyanın yeni adı '{yeni_dosya_adi}' oldu.")


In [ ]:
len(os.listdir("/content/drive/MyDrive/ImageSegmentationYOLOv8/all_info/labels/val"))

In [13]:
import os
len(os.listdir("/content/drive/MyDrive/all_info/images/train")) , len(os.listdir("/content/drive/MyDrive/all_info/images/val"))


(789, 198)

In [14]:

len(os.listdir("/content/drive/MyDrive/all_info/labels/train")) , len(os.listdir("/content/drive/MyDrive/all_info/labels/val"))

(789, 198)